In [ ]:
# ========================================
# Full CLIP Training: All 6 Shards (277k images)
# ========================================

import sys
sys.path.append('../')

import torch
from torch.utils.data import DataLoader, random_split
from src.dataset import NTIRETrainDataset, TransformSubset, get_transforms
from src.models import CLIPDetector, DINOv2Detector
import time
from datetime import datetime
import torch.nn as nn
from tqdm import tqdm
from sklearn.metrics import roc_auc_score

print("=" * 60)
print("FULL TRAINING - ALL 6 SHARDS")
print("=" * 60)
print(f"Started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 60)

# Config
BATCH_SIZE = 16
EPOCHS = 6  # Fixed 6 epochs, no early stopping
LR = 1e-4
ACCUMULATION_STEPS = 2  # Effective batch = 32
VAL_SPLIT = 0.1

# Device
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"\n💻 Device: {device}")

# Load dataset (ALL 6 shards) - NO transform yet
print("\n📂 Loading full dataset...")
train_transform, val_transform = get_transforms()

base_dataset = NTIRETrainDataset(
    '../data/train',
    shard_nums=None,  # All 6 shards
    transform=None  # No transform on base dataset
)

# Split indices (90/10)
val_size = int(len(base_dataset) * VAL_SPLIT)
train_size = len(base_dataset) - val_size

indices = list(range(len(base_dataset)))
train_indices, val_indices = random_split(
    indices,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

# Create subsets with DIFFERENT transforms
print("  Creating train subset with augmentation...")
train_dataset = TransformSubset(base_dataset, train_indices, train_transform)

print("  Creating val subset with minimal transforms...")
val_dataset = TransformSubset(base_dataset, val_indices, val_transform)

print(f"\n📊 Dataset Split:")
print(f"  Train: {len(train_dataset):,} images")
print(f"  Val:   {len(val_dataset):,} images")

# DataLoaders
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=False
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=False
)

# Estimate time
iterations_per_epoch = len(train_loader)
time_per_iteration = 1.5  # seconds (based on medium training)
time_per_epoch_min = (iterations_per_epoch * time_per_iteration) / 60
total_time_hours = (time_per_epoch_min * EPOCHS) / 60

print(f"\n⏱️  Training Estimate:")
print(f"  Iterations/epoch: {iterations_per_epoch:,}")
print(f"  Time/epoch: ~{time_per_epoch_min:.0f} minutes")
print(f"  Total time: ~{total_time_hours:.1f} hours")

estimated_finish = datetime.now().timestamp() + (total_time_hours * 3600)
finish_time = datetime.fromtimestamp(estimated_finish)
print(f"  Estimated finish: {finish_time.strftime('%Y-%m-%d %H:%M:%S')}")

# Model
#model = CLIPDetector()
model = DINOv2Detector()
print(f"\n🤖 Creating model {type(model).__name__}")

# Important warnings
print("\n" + "=" * 60)
print("⚠️  IMPORTANT - READ BEFORE STARTING")
print("=" * 60)
print(f"1. Training will take ~{total_time_hours:.0f} hours")
print("2. Mac will be busy - cannot use for other tasks")
print("3. DISABLE SLEEP MODE:")
print("   System Settings → Battery → Prevent auto sleep")
print("4. Keep Mac plugged in and charging")
print("5. Checkpoints saved every epoch (6 files total)")
print("6. If training fails, you keep completed epoch checkpoints")
print("=" * 60)

# Confirmation
print("\n" + "=" * 60)
response = input("Ready to start? Type 'YES' to continue: ")
if response.upper() != 'YES':
    print("Training cancelled.")
    sys.exit(0)

print("=" * 60)
print("🚀 TRAINING STARTED")
print("=" * 60)

# Modified training function to save every epoch
def train_full_model(model, train_loader, val_loader, epochs, lr, device, accumulation_steps):
    """
    Train model and save checkpoint EVERY epoch
    """
    
    model = model.to(device)
    
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = nn.BCEWithLogitsLoss()
    
    best_auc = 0.0
    history = {'train_loss': [], 'val_auc': [], 'epochs': []}
    
    # Open log file
    log_file = open('../checkpoints/full_training.log', 'w')
    
    def log(msg):
        """Print and save to file"""
        print(msg)
        log_file.write(msg + '\n')
        log_file.flush()
    
    log(f"Training started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    log("=" * 60)
    
    for epoch in range(epochs):
        epoch_start = time.time()
        
        log(f"\nEpoch {epoch+1}/{epochs}")
        log("-" * 60)
        
        # TRAINING
        model.train()
        train_loss = 0.0
        optimizer.zero_grad()
        
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")
        for i, (images, labels) in enumerate(pbar):
            images = images.to(device)
            labels = labels.float().to(device)
            
            outputs = model(images).squeeze()
            loss = criterion(outputs, labels) / accumulation_steps
            loss.backward()
            
            if (i + 1) % accumulation_steps == 0:
                optimizer.step()
                optimizer.zero_grad()
            
            train_loss += loss.item() * accumulation_steps
            pbar.set_postfix({'loss': f'{loss.item() * accumulation_steps:.4f}'})
        
        avg_loss = train_loss / len(train_loader)
        
        # VALIDATION
        model.eval()
        all_preds = []
        all_labels = []
        
        with torch.no_grad():
            for images, labels in tqdm(val_loader, desc="Validation"):
                images = images.to(device)
                outputs = model(images).squeeze()
                probs = torch.sigmoid(outputs)
                
                all_preds.extend(probs.cpu().numpy())
                all_labels.extend(labels.numpy())
        
        val_auc = roc_auc_score(all_labels, all_preds)
        
        # Update history
        history['train_loss'].append(avg_loss)
        history['val_auc'].append(val_auc)
        history['epochs'].append(epoch + 1)
        
        # Calculate epoch time
        epoch_time = time.time() - epoch_start
        remaining_epochs = epochs - (epoch + 1)
        remaining_time = remaining_epochs * epoch_time / 3600
        
        # Log results
        log(f"Epoch {epoch+1}/{epochs} Results:")
        log(f"  Train Loss: {avg_loss:.4f}")
        log(f"  Val AUC:    {val_auc:.4f}")
        log(f"  Epoch time: {epoch_time/60:.1f} minutes")
        log(f"  Remaining:  ~{remaining_time:.1f} hours")
        
        # Save checkpoint EVERY epoch
        checkpoint_path = f'../checkpoints/clip_full_epoch{epoch+1}.pth'
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'auc': val_auc,
            'loss': avg_loss,
            'history': history
        }, checkpoint_path)
        log(f"  ✓ Checkpoint saved: {checkpoint_path}")
        
        # Track best
        if val_auc > best_auc:
            best_auc = val_auc
            log(f"  🏆 New best AUC!")
        
        scheduler.step()
    
    log("\n" + "=" * 60)
    log(f"Training completed: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    log(f"Best AUC: {best_auc:.4f}")
    log("=" * 60)
    
    log_file.close()
    
    return model, history

# Train
model, history = train_full_model(
    model,
    train_loader,
    val_loader,
    epochs=EPOCHS,
    lr=LR,
    device=device,
    accumulation_steps=ACCUMULATION_STEPS
)

# Summary
print("\n" + "=" * 60)
print("✅ FULL TRAINING COMPLETED")
print("=" * 60)
print(f"Total epochs: {EPOCHS}")
print(f"Best val AUC: {max(history['val_auc']):.4f}")
print(f"Best epoch: {history['val_auc'].index(max(history['val_auc'])) + 1}")
print(f"\nCheckpoints saved:")
for i in range(1, EPOCHS+1):
    print(f"  - checkpoints/clip_full_epoch{i}.pth")
print(f"\nLog file: checkpoints/full_training.log")
print("=" * 60)

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/albumentations/check_version.py:147: UserWarning: Error fetching version info <urlopen error [SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1018)>
  data = fetch_version_info()


FULL CLIP TRAINING - ALL 6 SHARDS
Started: 2026-02-14 18:54:54

💻 Device: mps

📂 Loading full dataset...
Loaded 6 shards: 277643 images
  Real: 100000 | Fake: 177643
  Creating train subset with augmentation...
  Creating val subset with minimal transforms...

📊 Dataset Split:
  Train: 249,879 images
  Val:   27,764 images

⏱️  Training Estimate:
  Iterations/epoch: 15,618
  Time/epoch: ~390 minutes
  Total time: ~39.0 hours
  Estimated finish: 2026-02-16 09:57:36

🤖 Creating model...
Loading openai/clip-vit-base-patch16...


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch16
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✓ Backbone frozen (only classifier trainable)
✓ Parameters: 197,121 trainable / 149,817,858 total

⚠️  IMPORTANT - READ BEFORE STARTING
1. Training will take ~39 hours
2. Mac will be busy - cannot use for other tasks
3. DISABLE SLEEP MODE:
   System Settings → Battery → Prevent auto sleep
4. Keep Mac plugged in and charging
5. Checkpoints saved every epoch (6 files total)
6. If training fails, you keep completed epoch checkpoints

🚀 TRAINING STARTED
Training started: 2026-02-14 18:57:21

Epoch 1/6
------------------------------------------------------------


...

Epoch 4/6 Results:
  Train Loss: 0.2358
  Val AUC:    0.9865
  Epoch time: 211.7 minutes
  Remaining:  ~7.1 hours
  ✓ Checkpoint saved: ../checkpoints/clip_full_epoch4.pth
  🏆 New best AUC!

Epoch 5/6
------------------------------------------------------------


Epoch 5/6:   0%|          | 0/15618 [00:00<?, ?it/s]/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/albumentations/check_version.py:147: UserWarning: Error fetching version info <urlopen error [SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1018)>
  data = fetch_version_info()
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/albumentations/check_version.py:147: UserWarning: Error fetching version info <urlopen error [SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1018)>
  data = fetch_version_info()
Validation:   0%|          | 0/1736 [00:00<?, ?it/s]/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/albumentations/check_version.py:147: UserWarning: Error fetching version info <urlopen error [SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer ce

Epoch 5/6 Results:
  Train Loss: 0.2280
  Val AUC:    0.9874
  Epoch time: 212.0 minutes
  Remaining:  ~3.5 hours
  ✓ Checkpoint saved: ../checkpoints/clip_full_epoch5.pth
  🏆 New best AUC!

Epoch 6/6
------------------------------------------------------------


Epoch 6/6:   0%|          | 0/15618 [00:00<?, ?it/s]/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/albumentations/check_version.py:147: UserWarning: Error fetching version info <urlopen error [SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1018)>
  data = fetch_version_info()
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/albumentations/check_version.py:147: UserWarning: Error fetching version info <urlopen error [SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1018)>
  data = fetch_version_info()
Validation:   0%|          | 0/1736 [00:00<?, ?it/s]/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/albumentations/check_version.py:147: UserWarning: Error fetching version info <urlopen error [SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer ce

Epoch 6/6 Results:
  Train Loss: 0.2244
  Val AUC:    0.9877
  Epoch time: 211.8 minutes
  Remaining:  ~0.0 hours
  ✓ Checkpoint saved: ../checkpoints/clip_full_epoch6.pth
  🏆 New best AUC!

Training completed: 2026-02-15 16:09:41
Best AUC: 0.9877

✅ FULL TRAINING COMPLETED
Total epochs: 6
Best val AUC: 0.9877
Best epoch: 6

Checkpoints saved:
  - checkpoints/clip_full_epoch1.pth
  - checkpoints/clip_full_epoch2.pth
  - checkpoints/clip_full_epoch3.pth
  - checkpoints/clip_full_epoch4.pth
  - checkpoints/clip_full_epoch5.pth
  - checkpoints/clip_full_epoch6.pth

Log file: checkpoints/full_training.log
